# Pre Processing Dataset Lama

In [1]:
import pandas as pd
import nltk
import time
import re
import tqdm

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## Load Dataset

In [2]:
load_df_lama = pd.read_csv('/content/drive/MyDrive/NER nlp/Dataset/LEGALNER-POS-PREV-NEXT-200.csv')
load_df_lama = load_df_lama.fillna(method="ffill")
df_grouped = load_df_lama.groupby('doc').agg({
    'sentence': ' '.join,
    'word': ' '.join,
    'pos': ' '.join,
    'prev': ' '.join,
    'next': ' '.join,
    'tag': ' '.join
}).reset_index()

<ipython-input-2-a3e82cf8f73e>:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  load_df_lama = pd.read_csv('/content/drive/MyDrive/NER nlp/Dataset/LEGALNER-POS-PREV-NEXT-200.csv')


In [ ]:
df = df_grouped[["word", "tag"]]
df = df.rename(columns={'word': 'text','tag':'text-tags'})
df['text'] = df['text'].apply(lambda x: x.lower().split())
df['text-tags'] = df['text-tags'].apply(lambda x: x.split())
df

,text,text-tags
0,"[putusan, nomor, :, 2387, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
1,"[putusan, nomor, :, 538, /, pid, ., b, /, 2018...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
2,"[putusan, ., nomor, :, 2280, /, pid, ., b, /, ...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I..."
3,"[putusan, nomor, :, 1107, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
4,"[putusan, nomor, 216, /, pid, ., c, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
...,...,...
102,"[putusan, nomor, :, 724, /, pid, ., sus, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
103,"[putusan, nomor, 848, /, pid, ., b, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
104,"[putusan, nomor, :, 538, /, pid, ., b, /, 2014...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
105,"[putusan, pidana, nomor, :, 536, /, pid, ., b,...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I..."


## Cleaning Dataset

In [ ]:
def cleaning_text(dataset):
  for i, row in tqdm.tqdm(dataset.iterrows(), total=len(dataset), desc="Cleaning text"):
    dataset.at[i, "text"] = [word.replace("\ufeff", "") for word in row["text"]]
    data_text, data_tag = zip(*[(token, label) for token, label in zip(row["text"], row["text-tags"]) if token.strip() != ''])
    dataset.at[i, "text"] = list(data_text)
    dataset.at[i, "text-tags"] = list(data_tag)

  return dataset

cleaning_text(df)

Cleaning text: 100%|██████████| 107/107 [00:02<00:00, 42.56it/s]


,text,text-tags
0,"[putusan, nomor, :, 2387, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
1,"[putusan, nomor, :, 538, /, pid, ., b, /, 2018...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
2,"[putusan, ., nomor, :, 2280, /, pid, ., b, /, ...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I..."
3,"[putusan, nomor, :, 1107, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
4,"[putusan, nomor, 216, /, pid, ., c, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
...,...,...
102,"[putusan, nomor, :, 724, /, pid, ., sus, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
103,"[putusan, nomor, 848, /, pid, ., b, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
104,"[putusan, nomor, :, 538, /, pid, ., b, /, 2014...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
105,"[putusan, pidana, nomor, :, 536, /, pid, ., b,...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I..."


## Split token

In [ ]:
def split_token(dataset):
  for i, row in tqdm.tqdm(dataset.iterrows(), total=len(dataset), desc="Processing rows"):
    for index, (text, tag) in enumerate(zip(row["text"], row["text-tags"])):
      if tag != "O":
        token = re.findall(r'[^\/,\.():;]+|[\/\.,():;]', text)
        add_label = []
        for i in range(len(token)):
          if 'B' in tag and not add_label:
            add_label.append(tag)
          else:
            if token[i] != ';':
              add_label.append(tag.replace('B', 'I'))
            else:
              add_label.append('O')

        del row["text"][index]
        del row["text-tags"][index]
        row["text"][index:index] = token
        row["text-tags"][index:index] = add_label
      else:
        token = re.findall(r'[^\/,\.():;]+|[\/\.,():;]', text)
        add_label = []
        for i in range(len(token)):
          add_label.append('O')

        del row["text"][index]
        del row["text-tags"][index]
        row["text"][index:index] = token
        row["text-tags"][index:index] = add_label
  return dataset
split_token(df)

Processing rows: 100%|██████████| 107/107 [00:50<00:00,  2.10it/s]


,text,text-tags
0,"[putusan, nomor, :, 2387, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
1,"[putusan, nomor, :, 538, /, pid, ., b, /, 2018...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
2,"[putusan, ., nomor, :, 2280, /, pid, ., b, /, ...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I..."
3,"[putusan, nomor, :, 1107, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
4,"[putusan, nomor, 216, /, pid, ., c, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
...,...,...
102,"[putusan, nomor, :, 724, /, pid, ., sus, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
103,"[putusan, nomor, 848, /, pid, ., b, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
104,"[putusan, nomor, :, 538, /, pid, ., b, /, 2014...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
105,"[putusan, pidana, nomor, :, 536, /, pid, ., b,...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I..."


## Menambahkan kolom Doc

In [ ]:
doc_column = ['doc: ' + str(i + 1) for i in range(len(df))]
df['doc'] = doc_column
df

,text,text-tags,doc
0,"[putusan, nomor, :, 2387, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1
1,"[putusan, nomor, :, 538, /, pid, ., b, /, 2018...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 2
2,"[putusan, ., nomor, :, 2280, /, pid, ., b, /, ...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I...",doc: 3
3,"[putusan, nomor, :, 1107, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 4
4,"[putusan, nomor, 216, /, pid, ., c, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 5
...,...,...,...
102,"[putusan, nomor, :, 724, /, pid, ., sus, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 103
103,"[putusan, nomor, 848, /, pid, ., b, /, 2019, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 104
104,"[putusan, nomor, :, 538, /, pid, ., b, /, 2014...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 105
105,"[putusan, pidana, nomor, :, 536, /, pid, ., b,...","[O, O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I...",doc: 106


## Split data per Sentence

In [ ]:
new_rows = []
for index, row in df.iterrows():
    sentences = ' '.join(row['text']).split(';')
    for sentence in sentences:
        new_rows.append({
            'text': sentence.split(),
            'text-tags': row['text-tags'],
            'doc': row['doc']
        })
new_df = pd.DataFrame(new_rows)
new_df

,text,text-tags,doc
0,"[putusan, nomor, :, 2387, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1
1,"[tempat, lahir, :, jakarta]","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1
2,"[umur, /, tanggal, lahir, :, 35, tahun, /, 6, ...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1
3,"[jenis, kelamin, :, laki, -, laki]","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1
4,"[kebangsaan, :, indonesia]","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1
...,...,...,...
20110,"[menjatuhkan, pidana, kepada, terdakwa, oleh, ...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107
20111,"[menetapkan, masa, penahanan, yang, telah, dij...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107
20112,"[menetapkan, terdakwa, tetap, berada, dalam, t...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107
20113,"[membebankan, terdakwa, untuk, membayar, biaya...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107


## Menambahkan kolom Sentence

In [ ]:
new_df['sentence'] = new_df.index.map(lambda x: f'sentence: {x + 1:06}')
new_df

,text,text-tags,doc,sentence
0,"[putusan, nomor, :, 2387, /, pid, ., sus, /, 2...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1,sentence: 000001
1,"[tempat, lahir, :, jakarta]","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1,sentence: 000002
2,"[umur, /, tanggal, lahir, :, 35, tahun, /, 6, ...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1,sentence: 000003
3,"[jenis, kelamin, :, laki, -, laki]","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1,sentence: 000004
4,"[kebangsaan, :, indonesia]","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 1,sentence: 000005
...,...,...,...,...
20110,"[menjatuhkan, pidana, kepada, terdakwa, oleh, ...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107,sentence: 020111
20111,"[menetapkan, masa, penahanan, yang, telah, dij...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107,sentence: 020112
20112,"[menetapkan, terdakwa, tetap, berada, dalam, t...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107,sentence: 020113
20113,"[membebankan, terdakwa, untuk, membayar, biaya...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 107,sentence: 020114


## Tokenisasi per kata

In [ ]:
tokens = []
tags = []
docs = []
sentences = []
for i, row in new_df.iterrows():
    for token, tag in zip(row['text'], row['text-tags']):
        tokens.append(token)
        tags.append(tag)
        docs.append(row['doc'])
        sentences.append(row['sentence'])
tokenized_df = pd.DataFrame({
    'text': tokens,
    'text-tags': tags,
    'doc': docs,
    'sentence': sentences
})
tokenized_df

,text,text-tags,doc,sentence
0,putusan,O,doc: 1,sentence: 000001
1,nomor,O,doc: 1,sentence: 000001
2,:,O,doc: 1,sentence: 000001
3,2387,B_VERN,doc: 1,sentence: 000001
4,/,I_VERN,doc: 1,sentence: 000001
...,...,...,...,...
1029300,penuntut,O,doc: 107,sentence: 020115
1029301,umum,O,doc: 107,sentence: 020115
1029302,dan,O,doc: 107,sentence: 020115
1029303,terdakwa,O,doc: 107,sentence: 020115


## Menambahkan Kata Prev dan Next

In [ ]:
tokenized_df['prev'] = tokenized_df['text'].shift(1)
tokenized_df['next'] = tokenized_df['text'].shift(-1)
tokenized_df['prev'].fillna('.', inplace=True)
tokenized_df['next'].fillna('.', inplace=True)

In [ ]:
tokenized_df

,text,text-tags,doc,sentence,prev,next
0,putusan,O,doc: 1,sentence: 000001,.,nomor
1,nomor,O,doc: 1,sentence: 000001,putusan,:
2,:,O,doc: 1,sentence: 000001,nomor,2387
3,2387,B_VERN,doc: 1,sentence: 000001,:,/
4,/,I_VERN,doc: 1,sentence: 000001,2387,pid
...,...,...,...,...,...,...
1029300,penuntut,O,doc: 107,sentence: 020115,.,umum
1029301,umum,O,doc: 107,sentence: 020115,penuntut,dan
1029302,dan,O,doc: 107,sentence: 020115,umum,terdakwa
1029303,terdakwa,O,doc: 107,sentence: 020115,dan,.


In [ ]:
proces_data = tokenized_df.rename(columns={'text': 'word', 'text-tags': 'tag'}).reindex(columns=['doc','sentence', 'word', 'prev', 'next', 'tag'])
proces_data

,doc,sentence,word,prev,next,tag
0,doc: 1,sentence: 000001,putusan,.,nomor,O
1,doc: 1,sentence: 000001,nomor,putusan,:,O
2,doc: 1,sentence: 000001,:,nomor,2387,O
3,doc: 1,sentence: 000001,2387,:,/,B_VERN
4,doc: 1,sentence: 000001,/,2387,pid,I_VERN
...,...,...,...,...,...,...
1029300,doc: 107,sentence: 020115,penuntut,.,umum,O
1029301,doc: 107,sentence: 020115,umum,penuntut,dan,O
1029302,doc: 107,sentence: 020115,dan,umum,terdakwa,O
1029303,doc: 107,sentence: 020115,terdakwa,dan,.,O


In [ ]:
proces_data.to_csv('Dataset-Lama.csv', index=False)